In [9]:
# --- Imports & config ---
# Reads a CSV of (sieve size, percent passing) and reports the
# minimal discrete match (MDM) particle count for that GSD.
# Optionally scales the MDM up to a whole-sample particle count.
import numpy as np
import pandas as pd

from gsd_lib import GSD, MinimalPackingGenerator

# CSV: two columns -> column 0 = sieve size (mm), column 1 = percent passing.
# Percent passing may be given as 0-100 or 0-1; both are handled.
CSV_PATH = "gsd.csv"

# MDM generator settings (match literature_comparison.ipynb).
X_N_FACTOR = 0.5
TOL = 1e-2

# Whole-sample estimate (optional). Sizes are in mm, so SAMPLE_VOLUME must be
# in mm^3. Set SAMPLE_VOLUME = None to skip the whole-sample estimate.
# VOID_RATIO e = V_voids / V_solids (e.g. 0.4 for a dense sand).
SAMPLE_VOLUME = 149.236922  # e.g. np.pi * (105 / 2) ** 2 * 25  for a 105 x 25 mm cylinder
VOID_RATIO = 0.8

In [10]:
# --- Load CSV -> build GSD -> generate MDM ---
df = pd.read_csv(CSV_PATH)
sizes = df.iloc[:, 0].to_numpy(dtype=float)
passing = df.iloc[:, 1].to_numpy(dtype=float)

# Sort ascending by size and normalize percent passing to 0-100.
order = np.argsort(sizes)
sizes, passing = sizes[order], passing[order]
if passing.max() <= 1.0:
    passing = passing * 100.0

# A GSD must span 0% -> 100% passing. If fines pass the smallest sieve,
# anchor the curve with a 0%-passing point just below it.
if passing[0] > 0:
    sizes = np.insert(sizes, 0, sizes[0] / 2)
    passing = np.insert(passing, 0, 0.0)
# Largest sieve carries 100% passing (nothing retained above it).
passing[-1] = 100.0

# Percent retained per interval; trailing entry stays 0 (complete GSD).
retained = np.zeros(len(sizes))
retained[:-1] = np.diff(passing)

gsd = GSD(sizes=sizes, masses=retained)
mdm = MinimalPackingGenerator(gsd, x_n_factor=X_N_FACTOR, tol=TOL, flex=True).mps

N_mdm = int(sum(mdm.quantities))
print(f"MDM number (total particle count): {N_mdm:,}")
print(f"USCS classification: {gsd.uscs_name} ({gsd.uscs_symbol})")

# --- Whole-sample estimate ---
# One MDM occupies V_solids * (1 + e) including voids; the sample holds
# (sample_volume / that) MDMs, each contributing N_mdm particles.
if SAMPLE_VOLUME is not None:
    mdm_total_volume = mdm.total_volume * (1 + VOID_RATIO)
    mdm_per_sample = SAMPLE_VOLUME / mdm_total_volume
    N_sample = N_mdm * mdm_per_sample
    print(f"\nSample volume: {SAMPLE_VOLUME:,.0f} mm^3 (void ratio e = {VOID_RATIO})")
    print(f"MDMs per sample: {mdm_per_sample:,.1f}")
    print(f"Estimated particles in whole sample: {N_sample:,.0f}")

MDM number (total particle count): 2,016
USCS classification: poorly graded sand (SP)

Sample volume: 149 mm^3 (void ratio e = 0.8)
MDMs per sample: 10.2
Estimated particles in whole sample: 20,563
